# QuantFolio: Portfolio Optimization & ML-Enhanced Asset Allocation
### Markowitz Mean-Variance + Machine Learning on Real NSE Data

**What this notebook covers:**
1. Real stock data from Yahoo Finance (10 NSE large-caps)
2. Return & risk analysis with correlation structure
3. Classical Markowitz Efficient Frontier
4. Monte Carlo portfolio simulation (50,000 random portfolios)
5. ML-enhanced allocation (predict returns → optimize weights)
6. Strategy backtesting & performance comparison

---

## Step 0: Setup & Install Dependencies

In [ ]:
# Install packages (run once)
!pip install yfinance scikit-learn matplotlib seaborn scipy pandas numpy --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# This single line ensures charts display in Jupyter
%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Color palette
COLORS = ['#2E86C1', '#E74C3C', '#27AE60', '#8E44AD', '#E67E22',
          '#1ABC9C', '#F39C12', '#3498DB', '#E91E63', '#00BCD4']

print("All packages ready!")

---
## Step 1: Download Real NSE Stock Data

We fetch **3 years of daily prices** for 10 large-cap NSE stocks across different sectors, plus the NIFTY50 index as benchmark.

| Stock | Sector | Why included |
|-------|--------|-------------|
| RELIANCE | Conglomerate | Largest by market cap |
| TCS, INFY, WIPRO | IT | Export-driven, USD correlation |
| HDFCBANK, ICICIBANK, SBIN | Banking | Rate-sensitive, domestic economy |
| LT | Infrastructure | Capex cycle proxy |
| BAJFINANCE | NBFC | Consumer credit growth |
| MARUTI | Auto | Discretionary spending indicator |

In [ ]:
import yfinance as yf

# NSE tickers (Yahoo Finance uses .NS suffix)
TICKERS = {
    'RELIANCE': 'RELIANCE.NS',
    'TCS': 'TCS.NS',
    'INFY': 'INFY.NS',
    'HDFCBANK': 'HDFCBANK.NS',
    'ICICIBANK': 'ICICIBANK.NS',
    'SBIN': 'SBIN.NS',
    'WIPRO': 'WIPRO.NS',
    'LT': 'LT.NS',
    'BAJFINANCE': 'BAJFINANCE.NS',
    'MARUTI': 'MARUTI.NS',
}
BENCHMARK = '^NSEI'  # NIFTY50

START_DATE = '2021-01-01'
END_DATE = '2025-03-14'
RISK_FREE_RATE = 0.065  # 6.5% (India 10Y govt bond yield approx)

print("Downloading stock data from Yahoo Finance...")
print("=" * 55)

# Download all stocks
prices = pd.DataFrame()
for name, ticker in TICKERS.items():
    try:
        data = yf.download(ticker, start=START_DATE, end=END_DATE, progress=False, auto_adjust=True)
        if isinstance(data.columns, pd.MultiIndex):
            data.columns = data.columns.get_level_values(0)
        prices[name] = data['Close']
        print(f"  {name:12s} {len(data):5d} days")
    except Exception as e:
        print(f"  {name:12s} FAILED: {e}")

# Download NIFTY50 benchmark
try:
    nifty = yf.download(BENCHMARK, start=START_DATE, end=END_DATE, progress=False, auto_adjust=True)
    if isinstance(nifty.columns, pd.MultiIndex):
        nifty.columns = nifty.columns.get_level_values(0)
    prices['NIFTY50'] = nifty['Close']
    print(f"  {'NIFTY50':12s} {len(nifty):5d} days")
except:
    pass

# Drop any rows with missing data
prices.dropna(inplace=True)
print(f"\nClean dataset: {len(prices)} trading days, {prices.shape[1]} assets")
print(f"Date range: {prices.index[0].strftime('%Y-%m-%d')} to {prices.index[-1].strftime('%Y-%m-%d')}")

# Separate stock prices (for optimization) from benchmark
stock_names = [c for c in prices.columns if c != 'NIFTY50']
stock_prices = prices[stock_names]

prices.tail()

---
## Step 2: Exploratory Data Analysis

Before optimizing, we need to understand the return distributions, volatility, and correlation structure of our universe.

In [ ]:
# ── 2.1: Normalized Price Performance ──
fig, ax = plt.subplots(figsize=(14, 7))
for i, col in enumerate(stock_names):
    normalized = stock_prices[col] / stock_prices[col].iloc[0] * 100
    ax.plot(stock_prices.index, normalized, lw=1.5, color=COLORS[i], label=col)

if 'NIFTY50' in prices.columns:
    nifty_norm = prices['NIFTY50'] / prices['NIFTY50'].iloc[0] * 100
    ax.plot(prices.index, nifty_norm, lw=2.5, color='black', ls='--', label='NIFTY50 (Benchmark)')

ax.axhline(y=100, color='gray', ls=':', lw=0.5)
ax.set_title('Normalized Price Performance (Base = 100)', fontsize=16, fontweight='bold')
ax.set_ylabel('Indexed Price')
ax.legend(loc='upper left', fontsize=9, ncol=2)
plt.tight_layout()
plt.show()

In [ ]:
# ── 2.2: Daily Returns ──
returns = stock_prices.pct_change().dropna()

# Annualized metrics
ann_returns = returns.mean() * 252
ann_vol = returns.std() * np.sqrt(252)
sharpe = (ann_returns - RISK_FREE_RATE) / ann_vol

metrics = pd.DataFrame({
    'Ann. Return (%)': (ann_returns * 100).round(2),
    'Ann. Volatility (%)': (ann_vol * 100).round(2),
    'Sharpe Ratio': sharpe.round(3),
    'Max Daily Loss (%)': (returns.min() * 100).round(2),
    'Max Daily Gain (%)': (returns.max() * 100).round(2),
}).sort_values('Sharpe Ratio', ascending=False)

print("Stock Risk-Return Profile (Annualized)")
print("=" * 70)
print(metrics.to_string())

In [ ]:
# ── 2.3: Risk-Return Scatter ──
fig, ax = plt.subplots(figsize=(12, 8))
for i, stock in enumerate(stock_names):
    ax.scatter(ann_vol[stock]*100, ann_returns[stock]*100,
               s=200, color=COLORS[i], zorder=5, edgecolors='white', lw=2)
    ax.annotate(stock, (ann_vol[stock]*100, ann_returns[stock]*100),
                textcoords="offset points", xytext=(10, 5), fontsize=10, fontweight='bold')

ax.set_xlabel('Annualized Volatility (%)', fontsize=13)
ax.set_ylabel('Annualized Return (%)', fontsize=13)
ax.set_title('Risk-Return Profile of NSE Stocks', fontsize=16, fontweight='bold')
ax.axhline(y=RISK_FREE_RATE*100, color='gray', ls='--', lw=1, label=f'Risk-Free Rate ({RISK_FREE_RATE*100:.1f}%)')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── 2.4: Correlation Matrix ──
corr = returns.corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            annot=True, fmt='.2f', square=True, linewidths=0.8,
            cbar_kws={'shrink': 0.8}, ax=ax)
ax.set_title('Return Correlation Matrix', fontsize=16, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

# Identify lowest correlations (best for diversification)
corr_pairs = corr.where(~np.triu(np.ones_like(corr, dtype=bool))).stack()
print("\nBest diversification pairs (lowest correlation):")
for pair, val in corr_pairs.nsmallest(5).items():
    print(f"  {pair[0]:12s} - {pair[1]:12s} : {val:.3f}")

---
## Step 3: Markowitz Mean-Variance Optimization

**Modern Portfolio Theory (MPT)** — Harry Markowitz, 1952 (Nobel Prize 1990):
- For a given return level, there exists a portfolio with minimum risk
- The set of all such portfolios forms the **Efficient Frontier**
- Rational investors should only hold portfolios on this frontier

We find three key portfolios:
1. **Minimum Variance Portfolio** — lowest possible risk
2. **Maximum Sharpe Ratio Portfolio** — best risk-adjusted return
3. **Maximum Return Portfolio** — highest expected return for given risk tolerance

In [ ]:
# ══════════════════════════════════════════════════════
# Portfolio Optimization Functions
# ══════════════════════════════════════════════════════

n_assets = len(stock_names)
mean_returns = returns.mean() * 252        # annualized
cov_matrix = returns.cov() * 252           # annualized

def portfolio_performance(weights):
    """Calculate annualized return, volatility, and Sharpe ratio."""
    port_return = np.dot(weights, mean_returns)
    port_vol = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
    sharpe = (port_return - RISK_FREE_RATE) / port_vol
    return port_return, port_vol, sharpe

def neg_sharpe(weights):
    return -portfolio_performance(weights)[2]

def portfolio_volatility(weights):
    return portfolio_performance(weights)[1]

def portfolio_return(weights):
    return portfolio_performance(weights)[0]

# Constraints: weights sum to 1
constraints = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}

# Bounds: 0-100% per stock (long-only)
bounds = tuple((0, 1) for _ in range(n_assets))

# Initial guess: equal weight
init_weights = np.array([1/n_assets] * n_assets)

print(f"Optimizing portfolios with {n_assets} assets...")
print(f"Covariance matrix shape: {cov_matrix.shape}")

In [ ]:
# ── 3.1: Maximum Sharpe Ratio Portfolio ──
result_sharpe = minimize(neg_sharpe, init_weights, method='SLSQP',
                         bounds=bounds, constraints=constraints)
max_sharpe_weights = result_sharpe.x
ms_ret, ms_vol, ms_sharpe = portfolio_performance(max_sharpe_weights)

print("MAXIMUM SHARPE RATIO PORTFOLIO")
print("=" * 45)
print(f"  Expected Return  : {ms_ret*100:>8.2f}%")
print(f"  Volatility       : {ms_vol*100:>8.2f}%")
print(f"  Sharpe Ratio     : {ms_sharpe:>8.3f}")
print(f"\n  Weights:")
for name, w in zip(stock_names, max_sharpe_weights):
    if w > 0.01:
        print(f"    {name:12s} : {w*100:>6.2f}%")

In [ ]:
# ── 3.2: Minimum Variance Portfolio ──
result_minvol = minimize(portfolio_volatility, init_weights, method='SLSQP',
                         bounds=bounds, constraints=constraints)
min_vol_weights = result_minvol.x
mv_ret, mv_vol, mv_sharpe = portfolio_performance(min_vol_weights)

print("MINIMUM VARIANCE PORTFOLIO")
print("=" * 45)
print(f"  Expected Return  : {mv_ret*100:>8.2f}%")
print(f"  Volatility       : {mv_vol*100:>8.2f}%")
print(f"  Sharpe Ratio     : {mv_sharpe:>8.3f}")
print(f"\n  Weights:")
for name, w in zip(stock_names, min_vol_weights):
    if w > 0.01:
        print(f"    {name:12s} : {w*100:>6.2f}%")

In [ ]:
# ── 3.3: Efficient Frontier ──
# Trace the frontier by optimizing for min vol at each return level
target_returns = np.linspace(mean_returns.min(), mean_returns.max(), 80)
frontier_vols = []

for target in target_returns:
    cons = [
        {'type': 'eq', 'fun': lambda w: np.sum(w) - 1},
        {'type': 'eq', 'fun': lambda w, t=target: portfolio_return(w) - t},
    ]
    result = minimize(portfolio_volatility, init_weights, method='SLSQP',
                      bounds=bounds, constraints=cons)
    if result.success:
        frontier_vols.append(portfolio_volatility(result.x))
    else:
        frontier_vols.append(np.nan)

frontier_vols = np.array(frontier_vols)
print(f"Efficient frontier computed: {np.sum(~np.isnan(frontier_vols))} valid points")

In [ ]:
# ── 3.4: Monte Carlo Simulation (50,000 Random Portfolios) ──
n_portfolios = 50000
mc_returns = np.zeros(n_portfolios)
mc_vols = np.zeros(n_portfolios)
mc_sharpes = np.zeros(n_portfolios)
mc_weights = np.zeros((n_portfolios, n_assets))

np.random.seed(42)
for i in range(n_portfolios):
    w = np.random.dirichlet(np.ones(n_assets))  # random weights summing to 1
    mc_weights[i] = w
    ret, vol, sr = portfolio_performance(w)
    mc_returns[i] = ret
    mc_vols[i] = vol
    mc_sharpes[i] = sr

print(f"Simulated {n_portfolios:,} random portfolios")
print(f"  Best Sharpe found: {mc_sharpes.max():.3f}")
print(f"  Lowest vol found:  {mc_vols.min()*100:.2f}%")

In [ ]:
# ── 3.5: PLOT — Efficient Frontier with Monte Carlo Cloud ──
fig, ax = plt.subplots(figsize=(14, 9))

# Monte Carlo cloud
scatter = ax.scatter(mc_vols*100, mc_returns*100, c=mc_sharpes, cmap='viridis',
                     alpha=0.3, s=5, label='Random Portfolios')
plt.colorbar(scatter, ax=ax, label='Sharpe Ratio', shrink=0.8)

# Efficient Frontier line
valid = ~np.isnan(frontier_vols)
ax.plot(frontier_vols[valid]*100, target_returns[valid]*100,
        color='#E74C3C', lw=3, label='Efficient Frontier', zorder=10)

# Max Sharpe portfolio
ax.scatter(ms_vol*100, ms_ret*100, color='#E74C3C', s=300, marker='*',
           zorder=15, edgecolors='black', lw=1.5, label=f'Max Sharpe ({ms_sharpe:.3f})')

# Min Variance portfolio
ax.scatter(mv_vol*100, mv_ret*100, color='#2E86C1', s=300, marker='*',
           zorder=15, edgecolors='black', lw=1.5, label=f'Min Variance (Vol={mv_vol*100:.1f}%)')

# Individual stocks
for i, stock in enumerate(stock_names):
    ax.scatter(ann_vol[stock]*100, ann_returns[stock]*100,
               s=100, color=COLORS[i], zorder=12, edgecolors='white', lw=1.5)
    ax.annotate(stock, (ann_vol[stock]*100, ann_returns[stock]*100),
                textcoords="offset points", xytext=(8, 4), fontsize=8, fontweight='bold')

# Equal weight portfolio
eq_ret, eq_vol, eq_sr = portfolio_performance(init_weights)
ax.scatter(eq_vol*100, eq_ret*100, color='#F39C12', s=200, marker='D',
           zorder=15, edgecolors='black', lw=1.5, label=f'Equal Weight (SR={eq_sr:.3f})')

ax.set_xlabel('Annualized Volatility (%)', fontsize=13)
ax.set_ylabel('Annualized Return (%)', fontsize=13)
ax.set_title('Efficient Frontier & Portfolio Optimization (50,000 Simulations)', fontsize=16, fontweight='bold')
ax.legend(loc='upper left', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.6: Portfolio Weight Comparison ──
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, weights, title in zip(axes,
    [max_sharpe_weights, min_vol_weights, init_weights],
    ['Max Sharpe Portfolio', 'Min Variance Portfolio', 'Equal Weight Portfolio']):

    mask = weights > 0.01
    labels = np.array(stock_names)[mask]
    sizes = weights[mask] * 100

    wedges, texts, autotexts = ax.pie(sizes, labels=labels, autopct='%1.1f%%',
                                       colors=COLORS[:len(labels)], startangle=90,
                                       textprops={'fontsize': 9})
    for t in autotexts:
        t.set_fontsize(8)
    ax.set_title(title, fontweight='bold', fontsize=13)

plt.tight_layout()
plt.show()

---
## Step 4: ML-Enhanced Portfolio Allocation

Classical Markowitz uses **historical** returns and covariance. We can improve this by using ML to **predict forward returns**, then optimize using those predictions instead.

**Approach:**
1. Engineer features (momentum, volatility, mean reversion signals)
2. Train a model to predict next-month stock returns
3. Use predicted returns in the Markowitz optimizer
4. Compare ML-enhanced allocation vs classical allocation

In [ ]:
# ── 4.1: Feature Engineering for Return Prediction ──
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_absolute_error, r2_score

# Build features for each stock (monthly rebalancing)
monthly_prices = stock_prices.resample('ME').last()
monthly_returns = monthly_prices.pct_change().dropna()

def build_features(returns_series):
    """Build predictive features for a single stock's return series."""
    df = pd.DataFrame({'Return': returns_series})
    # Momentum features
    df['Mom_1m'] = df['Return'].shift(1)         # last month return
    df['Mom_3m'] = df['Return'].rolling(3).mean().shift(1)  # 3-month avg
    df['Mom_6m'] = df['Return'].rolling(6).mean().shift(1)  # 6-month avg
    # Volatility features
    df['Vol_3m'] = df['Return'].rolling(3).std().shift(1)
    df['Vol_6m'] = df['Return'].rolling(6).std().shift(1)
    # Mean reversion
    df['Deviation'] = (df['Return'].shift(1) - df['Return'].rolling(12).mean().shift(1))
    return df.dropna()

# Build feature matrices for all stocks
all_features = {}
for stock in stock_names:
    all_features[stock] = build_features(monthly_returns[stock])

sample = all_features[stock_names[0]]
print(f"Features per stock: {sample.shape[1] - 1}")
print(f"Months of data: {sample.shape[0]}")
print(f"\nFeature columns: {[c for c in sample.columns if c != 'Return']}")

In [ ]:
# ── 4.2: Train ML Models ──
# We train one model per stock, predicting next-month return

feature_cols = ['Mom_1m', 'Mom_3m', 'Mom_6m', 'Vol_3m', 'Vol_6m', 'Deviation']
predictions = {}
model_scores = {}

print("Training ML models (one per stock)...")
print("=" * 55)

for stock in stock_names:
    df = all_features[stock]
    X = df[feature_cols].values
    y = df['Return'].values

    # Time-series split: train on first 70%, test on last 30%
    split = int(len(X) * 0.7)
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    # Scale features
    scaler = RobustScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    # Gradient Boosting (best for tabular financial data)
    model = GradientBoostingRegressor(
        n_estimators=100, max_depth=3, learning_rate=0.1,
        subsample=0.8, random_state=42
    )
    model.fit(X_train_s, y_train)
    y_pred = model.predict(X_test_s)

    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    direction_acc = np.mean(np.sign(y_test) == np.sign(y_pred))

    model_scores[stock] = {'MAE': mae, 'R2': r2, 'Dir_Acc': direction_acc}
    predictions[stock] = {
        'model': model, 'scaler': scaler,
        'y_test': y_test, 'y_pred': y_pred,
        'last_pred': y_pred[-1]  # most recent prediction
    }

    print(f"  {stock:12s}  MAE={mae:.4f}  R2={r2:+.3f}  DirAcc={direction_acc:.1%}")

scores_df = pd.DataFrame(model_scores).T
print(f"\nAverage Direction Accuracy: {scores_df['Dir_Acc'].mean():.1%}")

In [ ]:
# ── 4.3: ML-Predicted Returns → Optimized Portfolio ──
# Use ML predictions as expected returns instead of historical averages

ml_expected_returns = np.array([predictions[s]['last_pred'] * 12 for s in stock_names])

# Keep the same covariance matrix (harder to predict covariance)
# Optimize using ML-predicted returns

def ml_portfolio_performance(weights):
    port_return = np.dot(weights, ml_expected_returns)
    port_vol = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
    sharpe = (port_return - RISK_FREE_RATE) / port_vol
    return port_return, port_vol, sharpe

def ml_neg_sharpe(weights):
    return -ml_portfolio_performance(weights)[2]

result_ml = minimize(ml_neg_sharpe, init_weights, method='SLSQP',
                     bounds=bounds, constraints=constraints)
ml_weights = result_ml.x
ml_ret, ml_vol, ml_sharpe = ml_portfolio_performance(ml_weights)

print("ML-ENHANCED PORTFOLIO (Predicted Returns)")
print("=" * 50)
print(f"  Expected Return  : {ml_ret*100:>8.2f}%")
print(f"  Volatility       : {ml_vol*100:>8.2f}%")
print(f"  Sharpe Ratio     : {ml_sharpe:>8.3f}")
print(f"\n  Weights:")
for name, w in zip(stock_names, ml_weights):
    if w > 0.01:
        print(f"    {name:12s} : {w*100:>6.2f}%")

---
## Step 5: Strategy Backtesting

We backtest 4 strategies over the test period to see which allocation approach performs best in practice:

1. **Equal Weight** — 10% each, rebalanced monthly
2. **Max Sharpe (Markowitz)** — classical optimization
3. **Min Variance** — risk-minimizing allocation
4. **ML-Enhanced** — ML-predicted returns fed into optimizer

In [ ]:
# ── 5.1: Backtest all strategies ──
# Use last 30% of data for backtesting
test_start = returns.index[int(len(returns) * 0.7)]
test_returns = returns.loc[test_start:]

strategies = {
    'Equal Weight': init_weights,
    'Max Sharpe': max_sharpe_weights,
    'Min Variance': min_vol_weights,
    'ML-Enhanced': ml_weights,
}

# Add NIFTY50 benchmark if available
if 'NIFTY50' in prices.columns:
    nifty_returns = prices['NIFTY50'].pct_change().dropna()
    nifty_test = nifty_returns.loc[test_start:]

# Calculate portfolio returns for each strategy
strategy_returns = pd.DataFrame(index=test_returns.index)
for name, weights in strategies.items():
    strategy_returns[name] = test_returns.dot(weights)

# Add NIFTY50 benchmark
if 'NIFTY50' in prices.columns:
    strategy_returns['NIFTY50'] = nifty_test.reindex(strategy_returns.index)

# Cumulative returns
cum_returns = (1 + strategy_returns).cumprod()

print(f"Backtest period: {test_returns.index[0].strftime('%Y-%m-%d')} to {test_returns.index[-1].strftime('%Y-%m-%d')}")
print(f"Trading days: {len(test_returns)}")

In [ ]:
# ── 5.2: Cumulative Performance Chart ──
fig, ax = plt.subplots(figsize=(14, 8))
style_map = {
    'Equal Weight': {'color': '#F39C12', 'lw': 2, 'ls': '-'},
    'Max Sharpe': {'color': '#E74C3C', 'lw': 2.5, 'ls': '-'},
    'Min Variance': {'color': '#2E86C1', 'lw': 2, 'ls': '-'},
    'ML-Enhanced': {'color': '#27AE60', 'lw': 2.5, 'ls': '-'},
    'NIFTY50': {'color': 'black', 'lw': 2, 'ls': '--'},
}

for col in cum_returns.columns:
    style = style_map.get(col, {'color': 'gray', 'lw': 1, 'ls': '-'})
    ax.plot(cum_returns.index, cum_returns[col], label=col, **style)

ax.axhline(y=1, color='gray', ls=':', lw=0.5)
ax.set_title('Strategy Backtest: Cumulative Returns', fontsize=16, fontweight='bold')
ax.set_ylabel('Growth of Rs 1')
ax.set_xlabel('Date')
ax.legend(fontsize=11)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.3: Strategy Performance Table ──
def compute_strategy_metrics(rets):
    ann_ret = rets.mean() * 252
    ann_vol = rets.std() * np.sqrt(252)
    sharpe = (ann_ret - RISK_FREE_RATE) / ann_vol
    cum = (1 + rets).cumprod()
    max_dd = (cum / cum.cummax() - 1).min()
    calmar = ann_ret / abs(max_dd) if max_dd != 0 else 0
    return {
        'Ann. Return (%)': round(ann_ret * 100, 2),
        'Ann. Volatility (%)': round(ann_vol * 100, 2),
        'Sharpe Ratio': round(sharpe, 3),
        'Max Drawdown (%)': round(max_dd * 100, 2),
        'Calmar Ratio': round(calmar, 3),
        'Final Value (Rs 1)': round((1 + rets).cumprod().iloc[-1], 4),
    }

perf = pd.DataFrame({name: compute_strategy_metrics(strategy_returns[name])
                      for name in strategy_returns.columns}).T
perf = perf.sort_values('Sharpe Ratio', ascending=False)

print("STRATEGY PERFORMANCE COMPARISON (Backtest Period)")
print("=" * 80)
print(perf.to_string())
print(f"\nBest strategy: {perf.index[0]} (Sharpe = {perf['Sharpe Ratio'].iloc[0]:.3f})")

In [ ]:
# ── 5.4: Drawdown Comparison ──
fig, ax = plt.subplots(figsize=(14, 6))
for col in ['Max Sharpe', 'ML-Enhanced', 'Equal Weight', 'Min Variance']:
    if col in cum_returns.columns:
        dd = (cum_returns[col] / cum_returns[col].cummax() - 1) * 100
        style = style_map.get(col, {'color': 'gray', 'lw': 1, 'ls': '-'})
        ax.plot(dd.index, dd, label=col, color=style['color'], lw=style['lw'])

ax.set_title('Drawdown Comparison (%)', fontsize=16, fontweight='bold')
ax.set_ylabel('Drawdown (%)')
ax.legend(fontsize=11)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.5: Monthly Returns Heatmap (Best Strategy) ──
best_strategy = perf.index[0]
best_monthly = strategy_returns[best_strategy].resample('ME').apply(lambda x: (1+x).prod()-1)
monthly_pivot = pd.DataFrame({
    'Year': best_monthly.index.year,
    'Month': best_monthly.index.month,
    'Return': best_monthly.values * 100
})
heatmap_data = monthly_pivot.pivot_table(index='Year', columns='Month', values='Return')
heatmap_data.columns = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'][:len(heatmap_data.columns)]

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(heatmap_data, cmap='RdYlGn', center=0, annot=True, fmt='.1f',
            linewidths=0.5, cbar_kws={'label': 'Return (%)'}, ax=ax)
ax.set_title(f'{best_strategy} — Monthly Returns Heatmap (%)', fontsize=14, fontweight='bold')
ax.set_ylabel('Year')
plt.tight_layout()
plt.show()

---
## Step 6: Final Summary

In [ ]:
# ── Final Summary ──
print("""
======================================================
  QUANTFOLIO — PORTFOLIO OPTIMIZATION COMPLETE
======================================================

  DATA
    10 NSE Large-Cap Stocks + NIFTY50 Benchmark
    Real market data from Yahoo Finance

  MARKOWITZ OPTIMIZATION
    Efficient Frontier: 80 points computed
    Monte Carlo: 50,000 random portfolios simulated""")

print(f"""
  OPTIMAL PORTFOLIOS
    Max Sharpe  : Return={ms_ret*100:.1f}%, Vol={ms_vol*100:.1f}%, SR={ms_sharpe:.3f}
    Min Variance: Return={mv_ret*100:.1f}%, Vol={mv_vol*100:.1f}%, SR={mv_sharpe:.3f}
    ML-Enhanced : Return={ml_ret*100:.1f}%, Vol={ml_vol*100:.1f}%, SR={ml_sharpe:.3f}

  ML MODELS
    Gradient Boosting per stock
    Avg Direction Accuracy: {scores_df['Dir_Acc'].mean():.1%}

  BACKTEST WINNER
    {perf.index[0]} (Sharpe = {perf['Sharpe Ratio'].iloc[0]:.3f})

  NEXT STEPS
    1. Run: streamlit run app.py  (interactive dashboard)
    2. Post on LinkedIn + GitHub
======================================================
""")